In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process import kernels
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression

from main import load_data
from imputation import transform_df
from outlier_detection import isolation_forest_outliers
from feature_selection import feature_importance_selection_keep_n


folder = 'data'
X_train_df, y_train_df, X_predict = load_data(
    f'{folder}/X_train.csv', f'{folder}/y_train.csv', f'{folder}/X_test.csv')


In [2]:
#scaling &imputation

#scaling
scaler = StandardScaler()
scaler.fit(X_train_df)
X_train_df = transform_df(X_train_df, scaler)

# #imputation
imputer = KNNImputer(n_neighbors=5)
imputer.fit(X_train_df)
X_train_df = transform_df(X_train_df, imputer)

In [4]:
#remove outliers
X_train_df, y_train_df = isolation_forest_outliers(X_train_df, y_train_df)


contamination:0.05; Number of outliers = 61


In [5]:
#feature selection
X_train_df, y_selector = feature_importance_selection_keep_n(X_train_df, y_train_df, 5)

c:\Users\domil\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1152: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
model = GaussianProcessRegressor()

rbf_kernels = [RBF(length_scale=length) for length in [1.0, 1.5, 2.0, 2.5, 3.0]]
rational_quadratic_kernels = [RationalQuadratic(length_scale=length) for length in [1.0, 1.5, 2.0, 2.5, 3.0]]

parameters = {
    'kernel': rbf_kernels + rational_quadratic_kernels,
}

gs = GridSearchCV(model, parameters, scoring='r2')
gs.fit(X_train_df, y_train_df)

In [31]:
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
model = KernelRidge()


parameters = {
    'kernel': ['poly'],
    'degree': [2,4],

}

gs = GridSearchCV(model, parameters, scoring='r2')
gs.fit(X_train_df, y_train_df)

GridSearchCV(estimator=KernelRidge(),
             param_grid={'degree': [2, 4], 'kernel': ['poly']}, scoring='r2')

In [32]:
from sklearn.metrics import r2_score

estimator = gs.best_estimator_
r2_score(y_train_df, estimator.predict(X_train_df))

0.3425030887352497

In [33]:
gs.best_score_

0.3102623723172054